# Figure 4: gaze context during high- and low-variability bouts
Section 2.8: select each participant's two highest and two lowest STCoV bouts, then calculate the fraction of valid gaze frames overlapping each category's bounding boxes. Overlapping categories remain non-exclusive. Average the two bout fractions within each participant before plotting; no statistical group tests.

Selection uses the available dense eye-tracking cache, which contains a previously selected subset of bouts with locally renumbered IDs. Its original selection cannot be fully reconstructed here. Thus this implements the stated method on available inputs, but an exact match to Figure 4 is not established. Equal STCoV values are ordered by BoutIndex; participants need four valid bouts. Figures use the source's 1088×1080 to 512×512 coordinate scaling and rounding.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
OUT  =  Path("results")
OUT.mkdir(exist_ok = True)
import matplotlib.pyplot as plt
import seaborn as sns

KEY = ['Participant','BoutIndex']
FKEY = KEY+['FrameIndex']
bouts = pd.read_parquet('derived_eye_tracking/bouts.parquet',columns = KEY+['BoutNumSteps','BoutStepTCV'])
frames = pd.read_parquet('derived_eye_tracking/frames.parquet',columns = FKEY+['FrameNumber','RawMaskPath','GazeX','GazeY'])
objects = pd.read_parquet('derived_eye_tracking/frame_objects.parquet',columns = FKEY+['ObjectName','BBoxX1','BBoxX2','BBoxY1','BBoxY2'])

assert not bouts.duplicated(KEY).any()
assert not frames.duplicated(FKEY).any()

valid = frames.loc[np.isfinite(frames[['GazeX','GazeY']]).all(axis = 1)].copy()
eligible = bouts.loc[bouts.Participant.str.startswith(('UFALL_PD','UFALL_CL')) & bouts.BoutNumSteps.gt(0) & np.isfinite(bouts.BoutStepTCV)].merge(valid[KEY].drop_duplicates(),on = KEY,validate = 'one_to_one')
selected = []

for participant,group in eligible.groupby('Participant',sort = True):
    if len(group) < 4: continue
    ordered = group.sort_values(['BoutStepTCV','BoutIndex'],kind = 'stable')
    selected.extend([ordered.head(2).assign(Variability = 'Low'),ordered.tail(2).assign(Variability = 'High')])

if not selected: raise ValueError('No participants have four eligible bouts')

selected = pd.concat(selected,ignore_index = True)
assert not selected.duplicated(KEY).any()

selected['Group'] = np.where(selected.Participant.str.startswith('UFALL_PD'),'PwP','Control')
valid = valid.merge(selected[KEY+['Variability','Group']],on = KEY,validate = 'many_to_one')
selected.to_csv(OUT/'figure4_selected_bouts.csv',index = False)
display(selected.groupby(['Group','Variability']).agg(Participants = ('Participant','nunique'),Bouts = ('BoutIndex','size')))

## Bounding-box overlap at frame level

In [ ]:
LABELS = {
    'Screens':['phone','laptop','television'],
    'Clutter':['clothes','slippers','shoes','box','bag','backpack','cable','container','toy','bin','paper','letter','book'],
    'Furniture':['chair','sofa','table','cabinet','shelf','stool','bed'],
    'Level change':['stairs','curb','step','steps','ramp'],
    'Social':['person','dog','cat'],
    'Structure':['wall','door','doorway','handrail'],
    'Vehicle':['car'],
    'Walkable surface':['floor','ground','pavement','grass','gravel','rug'],
}

label_to_category = {label:category for category,labels in LABELS.items() for label in labels}
coarse = {'Walkable surface':'Locomotor surface','Level change':'Locomotor surface',
        'Social':'Social/dynamic','Vehicle':'Social/dynamic',
        'Screens':'Non-locomotor','Clutter':'Non-locomotor','Furniture':'Non-locomotor','Structure':'Non-locomotor'}

joined = objects.merge(valid[FKEY+['GazeX','GazeY']],on = FKEY,validate = 'many_to_one').dropna(subset = ['BBoxX1','BBoxX2','BBoxY1','BBoxY2'])
unknown = sorted(set(joined.ObjectName.dropna())-set(label_to_category))

if unknown: raise ValueError(f'Unmapped object labels: {unknown}')

# Scaling both bounding boxes and gaze consistently; rounding matches the source notebook.
for col in ['GazeX','BBoxX1','BBoxX2']: 
    joined[col] = np.rint(joined[col]*512/1088)

for col in ['GazeY','BBoxY1','BBoxY2']: 
    joined[col] = np.rint(joined[col]*512/1080)

inside = (joined.GazeX.ge(joined[['BBoxX1','BBoxX2']].min(axis = 1)) & joined.GazeX.le(joined[['BBoxX1','BBoxX2']].max(axis = 1)) &
        joined.GazeY.ge(joined[['BBoxY1','BBoxY2']].min(axis = 1)) & joined.GazeY.le(joined[['BBoxY1','BBoxY2']].max(axis = 1)))

hits = joined.loc[inside,FKEY+['ObjectName']].copy()
hits['Granular'] = hits.ObjectName.map(label_to_category)
hits['Coarse'] = hits.Granular.map(coarse)

# Count a category at most once per frame, even if several of its objects overlap.
fractions = []
totals = valid.groupby(KEY).size().rename('Valid_frames')
for level,categories in [('Granular',list(LABELS)),('Coarse',['Locomotor surface','Non-locomotor','Social/dynamic'])]:
    counts = hits.drop_duplicates(FKEY+[level]).groupby(KEY+[level]).size().unstack(level)
    counts = counts.reindex(index = totals.index,columns = categories).fillna(0)
    table = counts.div(totals,axis = 0).rename_axis(columns = 'Category').reset_index().melt(id_vars = KEY,var_name = 'Category',value_name = 'Fraction')
    table['Level'] = level
    fractions.append(table)

bout_fractions = pd.concat(fractions,ignore_index = True).merge(selected[KEY+['Group','Variability']],on = KEY,validate = 'many_to_one')
assert bout_fractions.Fraction.between(0,1).all()

participant_fractions = bout_fractions.groupby(['Participant','Group','Variability','Level','Category'],as_index = False).Fraction.mean()
bout_fractions.to_csv(OUT/'figure4_bout_fractions.csv',index = False)
participant_fractions.to_csv(OUT/'figure4_participant_fractions.csv',index = False)
display(participant_fractions.groupby(['Group','Variability','Level','Category']).Fraction.agg(['count','mean','median']))

## Figure 4: descriptive participant distributions

In [ ]:
fig,axes = plt.subplots(2,2,figsize = (14,10),constrained_layout = True)
for col,variability in enumerate(['High','Low']):
    coarse_data = participant_fractions.loc[participant_fractions.Variability.eq(variability)&participant_fractions.Level.eq('Coarse')]
    sns.boxplot(data = coarse_data,x = 'Category',y = 'Fraction',hue = 'Group',order = ['Locomotor surface','Non-locomotor','Social/dynamic'],hue_order = ['Control','PwP'],ax = axes[0,col])
    axes[0,col].set_title(f'{chr(65+col)}  {variability} variability')
    granular = participant_fractions.loc[participant_fractions.Variability.eq(variability)&participant_fractions.Level.eq('Granular')&participant_fractions.Group.eq('PwP')]
    sns.boxplot(data = granular,x = 'Category',y = 'Fraction',order = list(LABELS),color = '#6596b5',ax = axes[1,col])
    axes[1,col].set_title(f'{chr(67+col)}  PwP: {variability.lower()} variability')
    axes[1,col].tick_params(axis = 'x',rotation = 40)
for ax in axes.flat:
    ax.set_ylim(0,1.02)
    ax.set_xlabel('')
    ax.set_ylabel('Mean fraction of valid gaze frames')
    
fig.savefig(OUT/'figure4.png',dpi = 160,bbox_inches = 'tight')
plt.show()